# Football Analysis with AI - Complete Pipeline

This notebook provides a comprehensive analysis of football video including:
- Player and ball detection using YOLO
- Player tracking with persistent IDs
- Team classification using deep embeddings
- Field detection and perspective transformation
- Voronoi diagram analysis
- Advanced heatmap visualizations
- Player movement statistics and distance calculations

## Section 1: Setup and Environment Configuration

### 1.1 - Check GPU Availability
Verify NVIDIA GPU availability using nvidia-smi command

In [ ]:
!nvidia-smi

### 1.2 - Import Display Utilities
Import IPython display module for visualization

In [ ]:
from IPython.display import Image

### 1.3 - Configure ONNX Runtime for CUDA
Set up ONNX Runtime to use CUDA ExecutionProvider for GPU acceleration

In [ ]:
import os
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"

## Section 2: Load and Visualize Video

### 2.1 - Load and Display First Video Frame
Load the source video file and extract the first frame for visualization

In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = "C:/Users/no3/Desktop/teste/projet_foot_rob/121364_0.mp4"

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

sv.plot_image(frame)

## Section 3: Player and Ball Detection

### 3.1 - Configure Player Detection Model Path
Set the path to the YOLO player detection model weights

In [ ]:
PLAYER_DETECTION_MODEL='C:/Users/no3/Desktop/teste/projet_foot_rob/best.pt'

### 3.2 - Check NumPy Version
Verify NumPy installation and version

In [ ]:
import numpy as np
print(np.__version__)

### 3.3 - Load YOLO Model and Perform Initial Detections
Load the YOLO detection model and run predictions on the video

In [ ]:
from ultralytics import YOLO

Model = YOLO('C:/Users/no3/Desktop/teste/weights/best.pt').cuda(device=0)

results = Model.predict('C:/Users/no3/Desktop/teste/projet_foot_rob/121364_0.mp4', save=True)
print (results[0])

for box in results[0].boxes:
    print(box)


### 3.4 - Set Source Video Path
Define the source video file path for analysis

In [ ]:
source_video= "C:/Users/no3/Desktop/teste/projet_foot_rob/121364_0.mp4"

### 3.5 - Visualize Player Detection with Bounding Boxes
Display detected players with bounding boxes and confidence scores

In [ ]:
import supervision as sv
import numpy as np
from ultralytics import YOLO

SOURCE_VIDEO_PATH = source_video
Model = YOLO('C:/Users/no3/Desktop/teste/projet_foot_rob/best.pt').cuda(device=0)

box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000')
)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

results = Model.predict(frame, conf=0.1)
result = results[0]

detections = sv.Detections.from_ultralytics(result)

labels = [
    f"Class ID: {class_id} {confidence:.2f}"
    for class_id, confidence in zip(detections.class_id, detections.confidence)
]

annotated_frame = frame.copy()
annotated_frame = box_annotator.annotate(
    scene=annotated_frame,
    detections=detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=detections,
    labels=labels)

sv.plot_image(annotated_frame)

### 3.6 - Separate Ball and Player Detection with Ellipse Markers
Detect players and ball separately, using different visual markers

In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = source_video
BALL_ID = 0

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

results = Model.predict(frame, conf=0.1)
result = results[0]

detections = sv.Detections.from_ultralytics(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections.class_id -= 1

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

## Section 4: Player Tracking and Team Classification

### 4.1 - Track Players with Unique Identifiers
Implement ByteTrack to assign persistent IDs to players across frames

In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = source_video
BALL_ID = 0

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)

tracker = sv.ByteTrack()
tracker.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

results = Model.predict(frame, conf=0.1)
result = results[0]

detections = sv.Detections.from_ultralytics(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections.class_id -= 1
all_detections = tracker.update_with_detections(detections=all_detections)

labels = [
    f"#{tracker_id}"
    for tracker_id
    in all_detections.tracker_id
]

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections,
    labels=labels)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

### 4.2 - Extract Player Image Crops for Classification
Extract individual player images from frames for team classification

In [ ]:
from tqdm import tqdm

SOURCE_VIDEO_PATH = source_video
PLAYER_ID = 2
STRIDE = 30

frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

crops = []
for frame in tqdm(frame_generator, desc='collecting crops'):
    results = Model.predict(frame, conf=0.1)
    result = results[0]

    detections = sv.Detections.from_ultralytics(result)
    detections = detections.with_nms(threshold=0.5, class_agnostic=True)
    detections = detections[detections.class_id == PLAYER_ID]
    players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
    crops += players_crops

### 4.3 - Display Grid of Player Crops
Visualize a grid of extracted player images

In [ ]:
sv.plot_images_grid(crops[:100], grid_size=(10, 10))

### 4.4 - Authenticate with Hugging Face Hub
Login to Hugging Face to access pre-trained models

In [ ]:
import os
from huggingface_hub import login
token = os.getenv('HF_TOKEN')
login(token=token)

### 4.5 - Load SIGLIP Vision Model for Embeddings
Load the SiGLIP model to extract image embeddings for players

In [ ]:
import torch
from transformers import AutoProcessor, SiglipVisionModel

SIGLIP_MODEL_PATH = 'google/siglip-base-patch16-224'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)
EMBEDDINGS_MODEL = SiglipVisionModel.from_pretrained(SIGLIP_MODEL_PATH).to(DEVICE)
EMBEDDINGS_PROCESSOR = AutoProcessor.from_pretrained(SIGLIP_MODEL_PATH, use_fast=True)

### 4.6 - Extract Embeddings from Player Crops
Compute SiGLIP embeddings for all player crops in batches

In [ ]:
import numpy as np
from more_itertools import chunked

BATCH_SIZE = 32

crops = [sv.cv2_to_pillow(crop) for crop in crops]
batches = chunked(crops, BATCH_SIZE)
data = []
with torch.no_grad():
    for batch in tqdm(batches, desc='embedding extraction'):
        inputs = EMBEDDINGS_PROCESSOR(images=batch, return_tensors="pt").to(DEVICE)
        outputs = EMBEDDINGS_MODEL(**inputs)
        embeddings = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
        data.append(embeddings)

data = np.concatenate(data)

### 4.7 - Initialize Dimensionality Reduction and Clustering
Prepare UMAP and KMeans for clustering embeddings

In [ ]:
import umap
from sklearn.cluster import KMeans

REDUCER = umap.UMAP(n_components=3)
CLUSTERING_MODEL = KMeans(n_clusters=2)

### 4.8 - Apply UMAP and K-Means to Embeddings
Reduce dimensionality and cluster players into teams

In [ ]:
projections = REDUCER.fit_transform(data)
clusters = CLUSTERING_MODEL.fit_predict(projections)

### 4.9 - Display Interactive 3D Projection with Image Viewer
Create an interactive 3D plot where clicking points shows player images

In [ ]:
import plotly.graph_objects as go
import numpy as np
from typing import Dict, List
from IPython.display import display, HTML
from PIL import Image
import base64
from io import BytesIO

def pil_image_to_data_uri(image: Image.Image) -> str:
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{img_str}"

def display_projections(
    labels: np.ndarray,
    projections: np.ndarray,
    images: List[Image.Image],
    show_legend: bool = False,
    show_markers_with_text: bool = True
) -> None:
    image_data_uris = {f"image_{i}": pil_image_to_data_uri(image) for i, image in enumerate(images)}
    image_ids = np.array([f"image_{i}" for i in range(len(images))])

    unique_labels = np.unique(labels)
    traces = []
    for unique_label in unique_labels:
        mask = labels == unique_label
        customdata_masked = image_ids[mask]
        trace = go.Scatter3d(
            x=projections[mask][:, 0],
            y=projections[mask][:, 1],
            z=projections[mask][:, 2],
            mode='markers+text' if show_markers_with_text else 'markers',
            text=labels[mask],
            customdata=customdata_masked,
            name=str(unique_label),
            marker=dict(size=8),
            hovertemplate="<b>class: %{text}</b><br>image ID: %{customdata}<extra></extra>"
        )
        traces.append(trace)

    all_axes = projections
    min_val = np.min(all_axes)
    max_val = np.max(all_axes)
    padding = (max_val - min_val) * 0.05
    axis_range = [min_val - padding, max_val + padding]

    fig = go.Figure(data=traces)
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', range=axis_range),
            yaxis=dict(title='Y', range=axis_range),
            zaxis=dict(title='Z', range=axis_range),
            aspectmode='cube'
        ),
        width=1000,
        height=1000,
        showlegend=show_legend
    )

    plotly_div = fig.to_html(full_html=False, include_plotlyjs=False, div_id="scatter-plot-3d")

    javascript_code = f"""
    <script>
        function displayImage(imageId) {{
            var imageElement = document.getElementById('image-display');
            var placeholderText = document.getElementById('placeholder-text');
            var imageDataURIs = {image_data_uris};
            imageElement.src = imageDataURIs[imageId];
            imageElement.style.display = 'block';
            placeholderText.style.display = 'none';
        }}

        var chartElement = document.getElementById('scatter-plot-3d');

        chartElement.on('plotly_click', function(data) {{
            var customdata = data.points[0].customdata;
            displayImage(customdata);
        }});
    </script>
    """

    html_template = f"""
    <!DOCTYPE html>
    <html>
        <head>
            <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
            <style>
                #image-container {{
                    position: fixed;
                    top: 0;
                    left: 0;
                    width: 200px;
                    height: 200px;
                    padding: 5px;
                    border: 1px solid #ccc;
                    background-color: white;
                    z-index: 1000;
                    box-sizing: border-box;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    text-align: center;
                }}
                #image-display {{
                    width: 100%;
                    height: 100%;
                    object-fit: contain;
                }}
            </style>
        </head>
        <body>
            {plotly_div}
            <div id="image-container">
                <img id="image-display" src="" alt="Selected image" style="display: none;" />
                <p id="placeholder-text">Click on a data entry to display an image</p>
            </div>
            {javascript_code}
        </body>
    </html>
    """

    display(HTML(html_template))

display_projections(clusters, projections, crops)

### 4.10 - Train Team Classifier on Player Crops
Train team classification model using the sports package

In [ ]:
import supervision as sv
from tqdm import tqdm
from sports.common.team import TeamClassifier

SOURCE_VIDEO_PATH = source_video
PLAYER_ID = 2
STRIDE = 30

frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

crops = []
for frame in tqdm(frame_generator, desc='collecting crops'):
    results = Model.predict(frame, conf=0.1)
    result = results[0]

    detections = sv.Detections.from_ultralytics(result)
    players_detections = detections[detections.class_id == PLAYER_ID]
    players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
    crops += players_crops

team_classifier = TeamClassifier(device="cuda")
team_classifier.fit(crops)

### 4.11 - Function to Assign Goalkeepers to Teams
Helper function to resolve goalkeeper team assignment based on spatial proximity

In [ ]:
import numpy as np
import supervision as sv

def resolve_goalkeepers_team_id(
    players: sv.Detections,
    goalkeepers: sv.Detections
) -> np.ndarray:
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0_centroid = players_xy[players.class_id == 0].mean(axis=0)
    team_1_centroid = players_xy[players.class_id == 1].mean(axis=0)
    goalkeepers_team_id = []
    for goalkeeper_xy in goalkeepers_xy:
        dist_0 = np.linalg.norm(goalkeeper_xy - team_0_centroid)
        dist_1 = np.linalg.norm(goalkeeper_xy - team_1_centroid)
        goalkeepers_team_id.append(0 if dist_0 < dist_1 else 1)

    return np.array(goalkeepers_team_id)

### 4.12 - Complete Detection and Annotation Pipeline
Combine all components to create full frame annotations with teams, tracking IDs, and ball

In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = source_video
BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(["#00FF55", '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(["#00FF44", '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex("#0026FF"),
    base=25,
    height=21,
    outline_thickness=1
)

tracker = sv.ByteTrack()
tracker.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

results = Model.predict(frame, conf=0.1)
result = results[0]

detections = sv.Detections.from_ultralytics(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections = tracker.update_with_detections(detections=all_detections)

goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
players_detections = all_detections[all_detections.class_id == PLAYER_ID]
referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
players_detections.class_id = team_classifier.predict(players_crops)

goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
    players_detections, goalkeepers_detections)

referees_detections.class_id -= 1

all_detections = sv.Detections.merge([
    players_detections, goalkeepers_detections, referees_detections])

labels = [
    f"#{tracker_id}"
    for tracker_id
    in all_detections.tracker_id
]

all_detections.class_id = all_detections.class_id.astype(int)

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections,
    labels=labels)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

### 4.13 - Display Source Video
Play the source video file for reference

In [ ]:
import supervision as sv
from IPython.display import Video, display

SOURCE_VIDEO_PATH = source_video

import os
if not os.path.exists(SOURCE_VIDEO_PATH):
    print(f"Error: Video file not found at {SOURCE_VIDEO_PATH}")
else:
    print("Displaying video using IPython.display.Video...")
    try:
        display(Video(SOURCE_VIDEO_PATH, embed=True))
    except Exception as e:
        print(f"Failed to display video: {e}")
        print("Please ensure the video file is not corrupted and the path is correct.")

## Section 5: Field Detection and Perspective Transform

### 5.1 - Load Field Keypoint Detection Model
Load YOLO model for detecting soccer field keypoints

In [ ]:
FIELD_DETECTION_MODEL = YOLO('C:/Users/no3/Desktop/teste/projet_foot_rob/football-pitch-detection.pt').cuda(device=0)

### 5.2 - Detect Soccer Field Keypoints
Detect field corner and center keypoints from video frame

In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = source_video

vertex_annotator = sv.VertexAnnotator(
    color=sv.Color.from_hex('#FF1493'),
    radius=8)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)
results = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
result = results[0]

key_points = sv.KeyPoints.from_ultralytics(result)

annotated_frame = frame.copy()
annotated_frame = vertex_annotator.annotate(
    scene=annotated_frame,
    key_points=key_points)

sv.plot_image(annotated_frame)

### 5.3 - Detect Field Keypoints with Confidence Filtering
Detect field keypoints from a specific frame with confidence threshold filtering

In [ ]:
import supervision as sv

SOURCE_VIDEO_PATH = source_video

vertex_annotator = sv.VertexAnnotator(
    color=sv.Color.from_hex('#FF1493'),
    radius=8)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, start=200)
frame = next(frame_generator)

results = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
result = results[0]

key_points = sv.KeyPoints.from_ultralytics(result)

filter = key_points.confidence[0] > 0.5
frame_reference_points = key_points.xy[0][filter]
frame_reference_key_points = sv.KeyPoints(
    xy=frame_reference_points[np.newaxis, ...])

annotated_frame = frame.copy()
annotated_frame = vertex_annotator.annotate(
    scene=annotated_frame,
    key_points=frame_reference_key_points)

sv.plot_image(annotated_frame)

### 5.4 - Draw Soccer Pitch Template
Display a standard soccer pitch template for visualization

In [ ]:
from sports.annotators.soccer import draw_pitch
from sports.configs.soccer import SoccerPitchConfiguration

CONFIG = SoccerPitchConfiguration()

annotated_frame = draw_pitch(CONFIG)

sv.plot_image(annotated_frame)

### 5.5 - Transform Frame to Pitch Coordinate System
Create ViewTransformer to project frame coordinates to pitch coordinates

In [ ]:
import numpy as np
import supervision as sv
from sports.common.view import ViewTransformer

SOURCE_VIDEO_PATH = source_video

edge_annotator = sv.EdgeAnnotator(
    color=sv.Color.from_hex('#00BFFF'),
    thickness=2, edges=CONFIG.edges)
vertex_annotator = sv.VertexAnnotator(
    color=sv.Color.from_hex('#FF1493'),
    radius=8)
vertex_annotator_2 = sv.VertexAnnotator(
    color=sv.Color.from_hex('#00BFFF'),
    radius=8)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, start=200)
frame = next(frame_generator)

results = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
result = results[0]

key_points = sv.KeyPoints.from_ultralytics(result)

filter = key_points.confidence[0] > 0.5
frame_reference_points = key_points.xy[0][filter]
frame_reference_key_points = sv.KeyPoints(
    xy=frame_reference_points[np.newaxis, ...])

pitch_reference_points = np.array(CONFIG.vertices)[filter]

transformer = ViewTransformer(
    source=pitch_reference_points,
    target=frame_reference_points
)

pitch_all_points = np.array(CONFIG.vertices)
frame_all_points = transformer.transform_points(points=pitch_all_points)

frame_all_key_points = sv.KeyPoints(xy=frame_all_points[np.newaxis, ...])

annotated_frame = frame.copy()
annotated_frame = edge_annotator.annotate(
    scene=annotated_frame,
    key_points=frame_all_key_points)
annotated_frame = vertex_annotator_2.annotate(
    scene=annotated_frame,
    key_points=frame_all_key_points)
annotated_frame = vertex_annotator.annotate(
    scene=annotated_frame,
    key_points=frame_reference_key_points)

sv.plot_image(annotated_frame)

### 5.6 - Retrain Team Classifier
Retrain team classification model with fresh crops

In [ ]:
import supervision as sv
from tqdm import tqdm
from sports.common.team import TeamClassifier

SOURCE_VIDEO_PATH = source_video
PLAYER_ID = 2
STRIDE = 30

frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

crops = []
for frame in tqdm(frame_generator, desc='collecting crops'):
    results = Model.predict(frame, conf=0.1)
    result = results[0]
    detections = sv.Detections.from_ultralytics(result)
    players_detections = detections[detections.class_id == PLAYER_ID]
    players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
    crops += players_crops

team_classifier = TeamClassifier(device="cuda")
team_classifier.fit(crops)

## Section 6: Pitch Visualization and Voronoi Analysis

### 6.1 - Define Voronoi Diagram Drawing Function
Implement custom Voronoi diagram function with smooth color blending between teams

In [ ]:
import cv2
from typing import Optional

def draw_pitch_voronoi_diagram_2(
    config: SoccerPitchConfiguration,
    team_1_xy: np.ndarray,
    team_2_xy: np.ndarray,
    team_1_color: sv.Color = sv.Color.RED,
    team_2_color: sv.Color = sv.Color.WHITE,
    opacity: float = 0.5,
    padding: int = 50,
    scale: float = 0.1,
    pitch: Optional[np.ndarray] = None
) -> np.ndarray:
    """
    Draws a Voronoi diagram on a soccer pitch representing the control areas of two
    teams with smooth color transitions.
    """
    if pitch is None:
        pitch = draw_pitch(
            config=config,
            padding=padding,
            scale=scale
        )

    scaled_width = int(config.width * scale)
    scaled_length = int(config.length * scale)

    voronoi = np.zeros_like(pitch, dtype=np.uint8)

    team_1_color_bgr = np.array(team_1_color.as_bgr(), dtype=np.uint8)
    team_2_color_bgr = np.array(team_2_color.as_bgr(), dtype=np.uint8)

    y_coordinates, x_coordinates = np.indices((
        scaled_width + 2 * padding,
        scaled_length + 2 * padding
    ))

    y_coordinates -= padding
    x_coordinates -= padding

    def calculate_distances(xy, x_coordinates, y_coordinates):
        return np.sqrt((xy[:, 0][:, None, None] * scale - x_coordinates) ** 2 +
                       (xy[:, 1][:, None, None] * scale - y_coordinates) ** 2)

    distances_team_1 = calculate_distances(team_1_xy, x_coordinates, y_coordinates)
    distances_team_2 = calculate_distances(team_2_xy, x_coordinates, y_coordinates)

    min_distances_team_1 = np.min(distances_team_1, axis=0)
    min_distances_team_2 = np.min(distances_team_2, axis=0)

    steepness = 15
    distance_ratio = min_distances_team_2 / np.clip(min_distances_team_1 + min_distances_team_2, a_min=1e-5, a_max=None)
    blend_factor = np.tanh((distance_ratio - 0.5) * steepness) * 0.5 + 0.5

    for c in range(3):
        voronoi[:, :, c] = (blend_factor * team_1_color_bgr[c] +
                            (1 - blend_factor) * team_2_color_bgr[c]).astype(np.uint8)

    overlay = cv2.addWeighted(voronoi, opacity, pitch, 1 - opacity, 0)

    return overlay

### 6.2 - Comprehensive Analysis with Player and Ball Projections
Perform complete analysis pipeline including player detection, team assignment, and pitch projections

In [ ]:
import supervision as sv
from sports.annotators.soccer import (
    draw_pitch,
    draw_points_on_pitch,
    draw_pitch_voronoi_diagram
)

SOURCE_VIDEO_PATH = source_video
BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=20, height=17
)

tracker = sv.ByteTrack()
tracker.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

results = Model.predict(frame, conf=0.1)
result = results[0]

detections = sv.Detections.from_ultralytics(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections = tracker.update_with_detections(detections=all_detections)

goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
players_detections = all_detections[all_detections.class_id == PLAYER_ID]
referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
players_detections.class_id = team_classifier.predict(players_crops)

goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
    players_detections, goalkeepers_detections)

referees_detections.class_id -= 1

all_detections = sv.Detections.merge([
    players_detections, goalkeepers_detections, referees_detections])

labels = [
    f"#{tracker_id}"
    for tracker_id
    in all_detections.tracker_id
]

all_detections.class_id = all_detections.class_id.astype(int)

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections,
    labels=labels)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

players_detections = sv.Detections.merge([
    players_detections, goalkeepers_detections
])

results = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
result = results[0]

key_points = sv.KeyPoints.from_ultralytics(result)

filter = key_points.confidence[0] > 0.5
frame_reference_points = key_points.xy[0][filter]
pitch_reference_points = np.array(CONFIG.vertices)[filter]

transformer = ViewTransformer(
    source=frame_reference_points,
    target=pitch_reference_points
)

frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
pitch_ball_xy = transformer.transform_points(points=frame_ball_xy)

players_xy = players_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
pitch_players_xy = transformer.transform_points(points=players_xy)

referees_xy = referees_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
pitch_referees_xy = transformer.transform_points(points=referees_xy)

annotated_frame = draw_pitch(CONFIG)
annotated_frame = draw_points_on_pitch(
    config=CONFIG,
    xy=pitch_ball_xy,
    face_color=sv.Color.WHITE,
    edge_color=sv.Color.BLACK,
    radius=10,
    pitch=annotated_frame)
annotated_frame = draw_points_on_pitch(
    config=CONFIG,
    xy=pitch_players_xy[players_detections.class_id == 0],
    face_color=sv.Color.from_hex('00BFFF'),
    edge_color=sv.Color.BLACK,
    radius=16,
    pitch=annotated_frame)
annotated_frame = draw_points_on_pitch(
    config=CONFIG,
    xy=pitch_players_xy[players_detections.class_id == 1],
    face_color=sv.Color.from_hex('FF1493'),
    edge_color=sv.Color.BLACK,
    radius=16,
    pitch=annotated_frame)
annotated_frame = draw_points_on_pitch(
    config=CONFIG,
    xy=pitch_referees_xy,
    face_color=sv.Color.from_hex('FFD700'),
    edge_color=sv.Color.BLACK,
    radius=16,
    pitch=annotated_frame)

sv.plot_image(annotated_frame)

annotated_frame = draw_pitch(CONFIG)
annotated_frame = draw_pitch_voronoi_diagram(
    config=CONFIG,
    team_1_xy=pitch_players_xy[players_detections.class_id == 0],
    team_2_xy=pitch_players_xy[players_detections.class_id == 1],
    team_1_color=sv.Color.from_hex('00BFFF'),
    team_2_color=sv.Color.from_hex('FF1493'),
    pitch=annotated_frame)

sv.plot_image(annotated_frame)

annotated_frame = draw_pitch(
    config=CONFIG,
    background_color=sv.Color.WHITE,
    line_color=sv.Color.BLACK
)
annotated_frame = draw_pitch_voronoi_diagram_2(
    config=CONFIG,
    team_1_xy=pitch_players_xy[players_detections.class_id == 0],
    team_2_xy=pitch_players_xy[players_detections.class_id == 1],
    team_1_color=sv.Color.from_hex('00BFFF'),
    team_2_color=sv.Color.from_hex('FF1493'),
    pitch=annotated_frame)
annotated_frame = draw_points_on_pitch(
    config=CONFIG,
    xy=pitch_ball_xy,
    face_color=sv.Color.WHITE,
    edge_color=sv.Color.WHITE,
    radius=8,
    thickness=1,
    pitch=annotated_frame)
annotated_frame = draw_points_on_pitch(
    config=CONFIG,
    xy=pitch_players_xy[players_detections.class_id == 0],
    face_color=sv.Color.from_hex('00BFFF'),
    edge_color=sv.Color.WHITE,
    radius=16,
    thickness=1,
    pitch=annotated_frame)
annotated_frame = draw_points_on_pitch(
    config=CONFIG,
    xy=pitch_players_xy[players_detections.class_id == 1],
    face_color=sv.Color.from_hex('FF1493'),
    edge_color=sv.Color.WHITE,
    radius=16,
    thickness=1,
    pitch=annotated_frame)

sv.plot_image(annotated_frame)

## Section 7: Video Output Generation

### 7.1 - Generate and Export Voronoi Blend Video
Process entire video and generate output with Voronoi diagram overlay

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

OUTPUT_VIDEO_PATH = "voronoi_blend_view.mp4"
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, video_info.fps, (video_info.width, video_info.height))

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
tracker = sv.ByteTrack()
tracker.reset()

for frame in tqdm(frame_generator, total=video_info.total_frames):
    results = Model.predict(frame, conf=0.1)
    result = results[0]
    detections = sv.Detections.from_ultralytics(result)

    ball_detections = detections[detections.class_id == BALL_ID]
    ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

    all_detections = detections[detections.class_id != BALL_ID]
    all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
    all_detections = tracker.update_with_detections(detections=all_detections)

    goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
    players_detections = all_detections[all_detections.class_id == PLAYER_ID]
    referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

    players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
    if len(players_crops) > 0:
        players_detections.class_id = team_classifier.predict(players_crops)
    if len(goalkeepers_detections) > 0:
        goalkeepers_detections.class_id = resolve_goalkeepers_team_id(players_detections, goalkeepers_detections)
    if len(referees_detections) > 0:
        referees_detections.class_id -= 1

    players_detections = sv.Detections.merge([players_detections, goalkeepers_detections])

    results_field = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
    result_field = results_field[0]
    key_points = sv.KeyPoints.from_ultralytics(result_field)
    filter = key_points.confidence[0] > 0.5
    if np.sum(filter) < 4:
        continue
    frame_reference_points = key_points.xy[0][filter]
    pitch_reference_points = np.array(CONFIG.vertices)[filter]

    from sports.common.view import ViewTransformer
    transformer = ViewTransformer(
        source=frame_reference_points,
        target=pitch_reference_points
    )

    frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_ball_xy = transformer.transform_points(points=frame_ball_xy)

    players_xy = players_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_players_xy = transformer.transform_points(points=players_xy)

    annotated_frame = draw_pitch(
        config=CONFIG,
        background_color=sv.Color.WHITE,
        line_color=sv.Color.BLACK
    )
    annotated_frame = draw_pitch_voronoi_diagram_2(
        config=CONFIG,
        team_1_xy=pitch_players_xy[players_detections.class_id == 0],
        team_2_xy=pitch_players_xy[players_detections.class_id == 1],
        team_1_color=sv.Color.from_hex('00BFFF'),
        team_2_color=sv.Color.from_hex('FF1493'),
        pitch=annotated_frame)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_ball_xy,
        face_color=sv.Color.WHITE,
        edge_color=sv.Color.WHITE,
        radius=8,
        thickness=1,
        pitch=annotated_frame)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_players_xy[players_detections.class_id == 0],
        face_color=sv.Color.from_hex('00BFFF'),
        edge_color=sv.Color.WHITE,
        radius=16,
        thickness=1,
        pitch=annotated_frame)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_players_xy[players_detections.class_id == 1],
        face_color=sv.Color.from_hex('FF1493'),
        edge_color=sv.Color.WHITE,
        radius=16,
        thickness=1,
        pitch=annotated_frame)

    annotated_frame = cv2.resize(annotated_frame, (video_info.width, video_info.height))
    annotated_frame_bgr = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
    writer.write(annotated_frame_bgr)

writer.release()
print(f"Video saved to: {OUTPUT_VIDEO_PATH}")

### 7.2 - Generate and Export Radar View Video
Generate top-down radar view of players and ball on the pitch

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

OUTPUT_VIDEO_PATH = "radar_view.mp4"
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, video_info.fps, (video_info.width, video_info.height))

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
tracker = sv.ByteTrack()
tracker.reset()

for frame in tqdm(frame_generator, total=video_info.total_frames):
    results = Model.predict(frame, conf=0.1)
    result = results[0]
    detections = sv.Detections.from_ultralytics(result)

    ball_detections = detections[detections.class_id == BALL_ID]
    ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

    all_detections = detections[detections.class_id != BALL_ID]
    all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
    all_detections = tracker.update_with_detections(detections=all_detections)

    goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
    players_detections = all_detections[all_detections.class_id == PLAYER_ID]
    referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

    players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
    if len(players_crops) > 0:
        players_detections.class_id = team_classifier.predict(players_crops)
    if len(goalkeepers_detections) > 0:
        goalkeepers_detections.class_id = resolve_goalkeepers_team_id(players_detections, goalkeepers_detections)
    if len(referees_detections) > 0:
        referees_detections.class_id -= 1

    players_detections = sv.Detections.merge([players_detections, goalkeepers_detections])

    results_field = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
    result_field = results_field[0]
    key_points = sv.KeyPoints.from_ultralytics(result_field)
    filter = key_points.confidence[0] > 0.5
    if np.sum(filter) < 4:
        continue
    frame_reference_points = key_points.xy[0][filter]
    pitch_reference_points = np.array(CONFIG.vertices)[filter]

    transformer = ViewTransformer(
        source=frame_reference_points,
        target=pitch_reference_points
    )

    frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_ball_xy = transformer.transform_points(points=frame_ball_xy)

    players_xy = players_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_players_xy = transformer.transform_points(points=players_xy)

    referees_xy = referees_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_referees_xy = transformer.transform_points(points=referees_xy)

    annotated_frame = draw_pitch(CONFIG)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_ball_xy,
        face_color=sv.Color.WHITE,
        edge_color=sv.Color.BLACK,
        radius=10,
        pitch=annotated_frame)
    if len(pitch_players_xy) > 0:
        annotated_frame = draw_points_on_pitch(
            config=CONFIG,
            xy=pitch_players_xy[players_detections.class_id == 0],
            face_color=sv.Color.from_hex('00BFFF'),
            edge_color=sv.Color.BLACK,
            radius=16,
            pitch=annotated_frame)
        annotated_frame = draw_points_on_pitch(
            config=CONFIG,
            xy=pitch_players_xy[players_detections.class_id == 1],
            face_color=sv.Color.from_hex('FF1493'),
            edge_color=sv.Color.BLACK,
            radius=16,
            pitch=annotated_frame)
    if len(pitch_referees_xy) > 0:
        annotated_frame = draw_points_on_pitch(
            config=CONFIG,
            xy=pitch_referees_xy,
            face_color=sv.Color.from_hex('FFD700'),
            edge_color=sv.Color.BLACK,
            radius=16,
            pitch=annotated_frame)

    annotated_frame = cv2.resize(annotated_frame, (video_info.width, video_info.height))
    annotated_frame_bgr = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
    writer.write(annotated_frame_bgr)

writer.release()
print(f"Video saved to: {OUTPUT_VIDEO_PATH}")

### 7.3 - Generate and Export Annotated Detection Video
Generate video with bounding boxes, tracking IDs, and team colors

In [ ]:
import cv2

OUTPUT_VIDEO_PATH = "output_annotated.mp4"
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, video_info.fps, (video_info.width, video_info.height))

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

for frame in frame_generator:
    results = Model.predict(frame, conf=0.1)
    result = results[0]
    detections = sv.Detections.from_ultralytics(result)

    ball_detections = detections[detections.class_id == BALL_ID]
    ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

    all_detections = detections[detections.class_id != BALL_ID]
    all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
    all_detections = tracker.update_with_detections(detections=all_detections)

    goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
    players_detections = all_detections[all_detections.class_id == PLAYER_ID]
    referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

    players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
    players_detections.class_id = team_classifier.predict(players_crops)
    goalkeepers_detections.class_id = resolve_goalkeepers_team_id(players_detections, goalkeepers_detections)
    referees_detections.class_id -= 1

    all_detections = sv.Detections.merge([players_detections, goalkeepers_detections, referees_detections])
    labels = [f"#{tracker_id}" for tracker_id in all_detections.tracker_id]
    all_detections.class_id = all_detections.class_id.astype(int)

    annotated_frame = frame.copy()
    annotated_frame = ellipse_annotator.annotate(scene=annotated_frame, detections=all_detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=all_detections, labels=labels)
    annotated_frame = triangle_annotator.annotate(scene=annotated_frame, detections=ball_detections)

    annotated_frame_bgr = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
    writer.write(annotated_frame_bgr)

writer.release()
print(f"Video saved to: {OUTPUT_VIDEO_PATH}")

## Section 8: Player Movement Analysis and Heatmaps

### 8.1 - Track Ball Movement Across Frames
Track ball trajectory and project onto pitch coordinates

In [ ]:
from collections import deque
import supervision as sv
from sports.annotators.soccer import draw_pitch, draw_points_on_pitch

SOURCE_VIDEO_PATH = source_video
BALL_ID = 0
MAXLEN = 5

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

path_raw = []
M = deque(maxlen=MAXLEN)

for frame in tqdm(frame_generator, total=video_info.total_frames):

    results = Model.predict(frame, conf=0.1)
    result = results[0]

    detections = sv.Detections.from_ultralytics(result)

    ball_detections = detections[detections.class_id == BALL_ID]
    ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

    results = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
    result = results[0]

    key_points = sv.KeyPoints.from_ultralytics(result)

    filter = key_points.confidence[0] > 0.5
    frame_reference_points = key_points.xy[0][filter]
    pitch_reference_points = np.array(CONFIG.vertices)[filter]

    transformer = ViewTransformer(
        source=frame_reference_points,
        target=pitch_reference_points
    )
    M.append(transformer.m)
    transformer.m = np.mean(np.array(M), axis=0)

    frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_ball_xy = transformer.transform_points(points=frame_ball_xy)

    path_raw.append(pitch_ball_xy)

### 8.2 - Prepare Ball Path Coordinates
Clean and prepare ball path data for visualization

In [ ]:
path = [
    np.empty((0, 2), dtype=np.float32) if coorinates.shape[0] >= 2 else coorinates
    for coorinates
    in path_raw
]

path = [coorinates.flatten() for coorinates in path]

### 8.3 - Visualize Ball Path on Soccer Pitch
Display the ball trajectory on the soccer pitch

In [ ]:
from sports.annotators.soccer import draw_paths_on_pitch

annotated_frame = draw_pitch(CONFIG)
annotated_frame = draw_paths_on_pitch(
    config=CONFIG,
    paths=[path],
    color=sv.Color.WHITE,
    pitch=annotated_frame)

sv.plot_image(annotated_frame)

### 8.4 - Define Outlier Removal Function
Function to remove outliers from ball path based on distance threshold

In [ ]:
from typing import List, Union

def replace_outliers_based_on_distance(
    positions: List[np.ndarray],
    distance_threshold: float
) -> List[np.ndarray]:
    last_valid_position: Union[np.ndarray, None] = None
    cleaned_positions: List[np.ndarray] = []

    for position in positions:
        if len(position) == 0:
            cleaned_positions.append(position)
        else:
            if last_valid_position is None:
                cleaned_positions.append(position)
                last_valid_position = position
            else:
                distance = np.linalg.norm(position - last_valid_position)
                if distance > distance_threshold:
                    cleaned_positions.append(np.array([], dtype=np.float64))
                else:
                    cleaned_positions.append(position)
                    last_valid_position = position

    return cleaned_positions

### 8.5 - Remove Outliers from Ball Path
Apply outlier removal to clean ball trajectory

In [ ]:
MAX_DISTANCE_THRESHOLD = 500

path = replace_outliers_based_on_distance(path, MAX_DISTANCE_THRESHOLD)

### 8.6 - Visualize Cleaned Ball Path
Display the cleaned ball trajectory without outliers

In [ ]:
from sports.annotators.soccer import draw_paths_on_pitch

annotated_frame = draw_pitch(CONFIG)
annotated_frame = draw_paths_on_pitch(
    config=CONFIG,
    paths=[path],
    color=sv.Color.WHITE,
    pitch=annotated_frame)

sv.plot_image(annotated_frame)

### 8.7 - Extract Player Position Data from Consecutive Frames
Extract movement data for players across consecutive video frames

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from sports.common.view import ViewTransformer

SOURCE_VIDEO_PATH = source_video
BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3
CONFIDENCE_THRESHOLD = 0.1
NMS_THRESHOLD = 0.5

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

consecutive_frames_data = []

tracker = sv.ByteTrack()
tracker.reset()

prev_frame_data = None
prev_frame_num = None

for frame_num, frame in tqdm(enumerate(frame_generator), total=video_info.total_frames, desc='Extracting consecutive frame pairs'):
    
    results = Model.predict(frame, conf=CONFIDENCE_THRESHOLD)
    result = results[0]
    detections = sv.Detections.from_ultralytics(result)
    
    all_detections = detections[detections.class_id != BALL_ID]
    all_detections = all_detections.with_nms(threshold=NMS_THRESHOLD, class_agnostic=True)
    
    all_detections = tracker.update_with_detections(detections=all_detections)
    
    goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
    players_detections = all_detections[all_detections.class_id == PLAYER_ID]
    
    if len(players_detections) > 0:
        players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
        players_detections.class_id = team_classifier.predict(players_crops)
    
    if len(goalkeepers_detections) > 0 and len(players_detections) > 0:
        goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
            players_detections, goalkeepers_detections)
    
    if len(goalkeepers_detections) > 0:
        all_players = sv.Detections.merge([players_detections, goalkeepers_detections])
    else:
        all_players = players_detections
    
    results_field = FIELD_DETECTION_MODEL.predict(frame, conf=0.3)
    result_field = results_field[0]
    key_points = sv.KeyPoints.from_ultralytics(result_field)
    
    filter_keypoints = key_points.confidence[0] > 0.5
    if np.sum(filter_keypoints) < 4:
        continue
    
    frame_reference_points = key_points.xy[0][filter_keypoints]
    pitch_reference_points = np.array(CONFIG.vertices)[filter_keypoints]
    
    transformer = ViewTransformer(
        source=frame_reference_points,
        target=pitch_reference_points
    )
    
    if len(all_players) > 0:
        frame_players_xy = all_players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pitch_players_xy = transformer.transform_points(points=frame_players_xy)
        
        current_frame_data = {
            'frame_num': frame_num,
            'players': []
        }
        
        for tracker_id, team_id, xy in zip(all_players.tracker_id, all_players.class_id, pitch_players_xy):
            current_frame_data['players'].append({
                'tracker_id': int(tracker_id),
                'team': int(team_id),
                'xy': xy
            })
        
        if prev_frame_data is not None:
            for player_prev in prev_frame_data['players']:
                for player_curr in current_frame_data['players']:
                    if (player_prev['tracker_id'] == player_curr['tracker_id'] and 
                        player_prev['team'] == player_curr['team']):
                        consecutive_frames_data.append({
                            'player_idx': player_prev['tracker_id'],
                            'team': player_prev['team'],
                            'xy_start': f"{player_prev['xy'][0]:.2f},{player_prev['xy'][1]:.2f}",
                            'num_frame_start': prev_frame_num,
                            'xy_end': f"{player_curr['xy'][0]:.2f},{player_curr['xy'][1]:.2f}",
                            'num_frame_end': frame_num,
                            'distance': np.linalg.norm(player_prev['xy'] - player_curr['xy'])
                        })
        
        prev_frame_data = current_frame_data
        prev_frame_num = frame_num

df_consecutive = pd.DataFrame(consecutive_frames_data)

if len(df_consecutive) > 0:
    df_consecutive = df_consecutive.sort_values(['num_frame_start', 'distance']).reset_index(drop=True)

print(f"\nExtracted {len(df_consecutive)} consecutive frame pair movements")
print("\nFirst few rows:")
print(df_consecutive.head(15))

### 8.8 - Save Consecutive Frame Position Data to CSV
Export player position data to CSV file

In [ ]:
output_csv_consecutive = "player_positions_consecutive_frames.csv"

df_export = df_consecutive.drop('distance', axis=1)
df_export.to_csv(output_csv_consecutive, index=False)

print(f"\nPlayer positions saved to: {output_csv_consecutive}")
print(f"\nDataFrame info:")
print(df_export.info())
print(f"\nTeam distribution:")
print(df_export['team'].value_counts())
print(f"\nMovements per frame (first 20):")
print(df_export.groupby('num_frame_start').size().head(20))
print(f"\nSample data:")
print(df_export.head(20))

### 8.9 - Display Frame and Movement Statistics
Show comprehensive statistics about frame processing and player movements

In [ ]:
print(f"Total frame pairs: {len(df_export)}")
print(f"Total frames processed: {df_export['num_frame_end'].max() + 1}")
print(f"\nTeam 0 movements: {len(df_export[df_export['team'] == 0])}")
print(f"Team 1 movements: {len(df_export[df_export['team'] == 1])}")
print(f"\nAverage movements per frame:")
print(df_export.groupby('num_frame_start').size().describe())

### 8.10 - Calculate Player Total Distance Traveled
Compute total distance traveled by a specific player

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('player_positions_consecutive_frames.csv', sep=';')

player_id = 1
player_data = df[df['player_idx'] == player_id].copy()

distances = []
for idx, row in player_data.iterrows():
    xy_start = np.array(list(map(float, row['xy_start'].split(','))))
    xy_end = np.array(list(map(float, row['xy_end'].split(','))))
    
    distance = np.linalg.norm(xy_end - xy_start)
    distances.append(distance)

distances = np.array(distances)

pixel_to_meter = 0.104
distances_m = distances * pixel_to_meter

total_distance = np.sum(distances_m)
avg_distance_per_frame = np.mean(distances_m)

print(f"--- DISTANCE STATISTICS FOR PLAYER {player_id} ---")
print(f"Total distance traveled: {total_distance:.2f} m")
print(f"Average distance per interval: {avg_distance_per_frame:.2f} m")
print(f"Number of intervals: {len(distances)}")
print(f"Maximum distance (interval): {np.max(distances_m):.2f} m")
print(f"Minimum distance (interval): {np.min(distances_m):.2f} m")

### 8.11 - Calculate Player Average Velocity
Compute velocity statistics for player movement

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('player_positions_consecutive_frames.csv', sep=';')

player_id = 1
player_data = df[df['player_idx'] == player_id].copy()

distances = []
for idx, row in player_data.iterrows():
    xy_start = np.array(list(map(float, row['xy_start'].split(','))))
    xy_end = np.array(list(map(float, row['xy_end'].split(','))))
    
    distance = np.linalg.norm(xy_end - xy_start)
    distances.append(distance)

distances = np.array(distances)

fps = 25
time_per_frame = 1 / fps
pixel_to_meter = 0.104

velocities = (distances * pixel_to_meter) / time_per_frame

avg_velocity = np.mean(velocities)
max_velocity = np.max(velocities)
min_velocity = np.min(velocities)

print(f"--- VELOCITY STATISTICS FOR PLAYER {player_id} ---")
print(f"Average velocity: {avg_velocity:.2f} m/s")
print(f"Maximum velocity: {max_velocity:.2f} m/s")
print(f"Minimum velocity: {min_velocity:.2f} m/s")
print(f"Number of measurements: {len(velocities)}")

### 8.12 - Generate Player Movement Density Heatmap
Create heatmap showing player presence density on the pitch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

df = pd.read_csv('player_positions_consecutive_frames.csv', sep=';')

player_id = 1
player_data = df[df['player_idx'] == player_id].copy()

coords = []
for idx, row in player_data.iterrows():
    xy_start = list(map(float, row['xy_start'].split(',')))
    coords.append(xy_start)
    xy_end = list(map(float, row['xy_end'].split(',')))
    coords.append(xy_end)

coords = np.array(coords)

fig, ax = plt.subplots(figsize=(14, 10))

x_min, x_max = 0, 8000
y_min, y_max = 0, 7000

grid_size = 100
x_grid = np.linspace(x_min, x_max, grid_size)
y_grid = np.linspace(y_min, y_max, grid_size)
X, Y = np.meshgrid(x_grid, y_grid)

from scipy.stats import gaussian_kde
positions = np.vstack([coords[:, 0], coords[:, 1]])
if len(coords) > 1:
    kde = gaussian_kde(positions)
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    Z = gaussian_filter(Z, sigma=2)
else:
    Z = np.zeros_like(X)

contour = ax.contourf(X, Y, Z, levels=20, cmap='hot')
scatter = ax.scatter(coords[:, 0], coords[:, 1], s=20, alpha=0.3, c='cyan')

cbar = plt.colorbar(contour, ax=ax)
cbar.set_label('Presence density')

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
ax.set_title(f'Player {player_id} Movement Density Heatmap')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"Positions recorded for player {player_id}: {len(coords)}")

### 8.13 - Visualize Player Positions on Soccer Pitch
Display all player positions on the soccer pitch visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
from sports.annotators.soccer import draw_pitch, draw_points_on_pitch
from sports.configs.soccer import SoccerPitchConfiguration

df = pd.read_csv('player_positions_consecutive_frames.csv', sep=';')

player_id = 1
player_data = df[df['player_idx'] == player_id].copy()

coords = []
for idx, row in player_data.iterrows():
    xy_start = list(map(float, row['xy_start'].split(',')))
    coords.append(xy_start)
    xy_end = list(map(float, row['xy_end'].split(',')))
    coords.append(xy_end)

coords = np.array(coords)

config = SoccerPitchConfiguration()

coords_normalized = coords.copy()

pitch_with_points = draw_points_on_pitch(
    config=config,
    xy=coords_normalized,
    face_color=sv.Color(255, 0, 0),
    edge_color=sv.Color(255, 255, 255),
    radius=8,
    thickness=2,
    padding=50,
    scale=0.1
)

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(pitch_with_points)
ax.set_title(f'Player {player_id} Positions on Pitch')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Positions for player {player_id}: {len(coords)}")
print(f"Pitch dimensions: {config.length} cm x {config.width} cm")
print(f"Position min: x={coords[:, 0].min():.0f}, y={coords[:, 1].min():.0f}")
print(f"Position max: x={coords[:, 0].max():.0f}, y={coords[:, 1].max():.0f}")

### 8.14 - Generate Advanced Temperature Heatmap of Player Movements
Create advanced heatmap with temperature coloring of player movement intensity

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
from sports.annotators.soccer import draw_pitch, draw_paths_on_pitch
from sports.configs.soccer import SoccerPitchConfiguration
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter
import cv2

df = pd.read_csv('player_positions_consecutive_frames.csv', sep=';')

player_id = 1
player_data = df[df['player_idx'] == player_id].copy()

config = SoccerPitchConfiguration()

coords = []
for idx, row in player_data.iterrows():
    xy_start = list(map(float, row['xy_start'].split(',')))
    coords.append(xy_start)
    xy_end = list(map(float, row['xy_end'].split(',')))
    coords.append(xy_end)

coords = np.array(coords)

pitch = draw_pitch(
    config=config,
    background_color=sv.Color(34, 139, 34),
    line_color=sv.Color.WHITE,
    padding=50,
    scale=0.1
)

scaled_length = int(config.length * 0.1)
scaled_width = int(config.width * 0.1)
padding = 50

x = np.linspace(0, config.length, scaled_length)
y = np.linspace(0, config.width, scaled_width)
X, Y = np.meshgrid(x, y)

positions = np.vstack([coords[:, 0], coords[:, 1]])
if len(coords) > 1:
    kde = gaussian_kde(positions)
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    Z = gaussian_filter(Z, sigma=3)
else:
    Z = np.zeros_like(X)

Z_normalized = (Z - Z.min()) / (Z.max() - Z.min() + 1e-8)

heatmap_img = np.zeros((scaled_width, scaled_length, 3), dtype=np.uint8)

for i in range(scaled_width):
    for j in range(scaled_length):
        intensity = Z_normalized[i, j]
        if intensity > 0.7:
            heatmap_img[i, j] = [0, 0, int(255 * intensity)]
        elif intensity > 0.4:
            heatmap_img[i, j] = [0, int(255 * intensity), 255]
        elif intensity > 0:
            heatmap_img[i, j] = [int(255 * intensity), 200, 100]

heatmap_img = cv2.GaussianBlur(heatmap_img, (15, 15), 0)

heatmap_resized = cv2.resize(heatmap_img, (scaled_length + 2*padding, scaled_width + 2*padding))

alpha = 0.6
pitch_with_heatmap = cv2.addWeighted(
    pitch, 
    1 - alpha, 
    cv2.cvtColor(heatmap_resized, cv2.COLOR_BGR2RGB), 
    alpha, 
    0
)

for point in coords:
    scaled_x = int(point[0] * 0.1) + padding
    scaled_y = int(point[1] * 0.1) + padding
    if 0 <= scaled_x < pitch_with_heatmap.shape[1] and 0 <= scaled_y < pitch_with_heatmap.shape[0]:
        cv2.circle(pitch_with_heatmap, (scaled_x, scaled_y), 4, (255, 0, 0), -1)

fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(pitch_with_heatmap)
ax.set_title(f'Temperature Heatmap of Player {player_id} Movements', fontsize=16, fontweight='bold')
ax.axis('off')

sm = plt.cm.ScalarMappable(cmap=plt.cm.cool, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Movement Density', fontsize=12)

plt.tight_layout()
plt.show()

print(f"--- TEMPERATURE HEATMAP FOR PLAYER {player_id} ---")
print(f"Positions: {len(coords)}")
print(f"Maximum density: {Z.max():.6f}")
print(f"Minimum density: {Z.min():.6f}")
print(f"Hot zones (red): High intensity areas")
print(f"Warm zones (orange/yellow): Medium intensity areas")
print(f"Cool zones (cyan): Low intensity areas")